In [4]:
import os, glob, numpy as np, pandas as pd, psycopg, pyarrow.parquet as pq
from dotenv import load_dotenv
load_dotenv("../.env")
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True, connect_timeout=10)

part = sorted(glob.glob("../data/embeddings/part-*.parquet"))[0]
t = pq.read_table(part)
ids = t.column("recommendationid").to_numpy()
vecs = t.column("embedding").combine_chunks().flatten().to_numpy().reshape(-1, 384)
norms = np.linalg.norm(vecs, axis=1)
print(part, vecs.shape, f"norms: min {norms.min():.4f}, max {norms.max():.4f}")   # both should be ~1.0000

../data/embeddings\part-205903981.parquet (10000, 384) norms: min 1.0000, max 1.0000


In [5]:
texts = pd.read_sql("SELECT recommendationid, review FROM reviews WHERE recommendationid = ANY(%(ids)s)",
                    conn, params={"ids": ids.tolist()}).set_index("recommendationid").loc[ids, "review"]

C:\Users\dillo\AppData\Local\Temp\ipykernel_11376\1527778897.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  texts = pd.read_sql("SELECT recommendationid, review FROM reviews WHERE recommendationid = ANY(%(ids)s)",


In [6]:
crash = [i for i, r in enumerate(texts) if "crash" in r.lower() and 40 < len(r) < 300]
q = crash[0]
sims = vecs @ vecs[q]                       # vectors are unit length, so dot product = cosine similarity
top = np.argsort(-sims)[1:6]                # [0] is the review itself
print(f"QUERY: {texts.iloc[q][:200]!r}\n")
for j in top:
    print(f"{sims[j]:.2f}  {texts.iloc[j][:200]!r}")
print(f"\nrandom pairs, for comparison: mean similarity {np.mean(vecs[np.random.default_rng(0).integers(0, len(vecs), 1000)] @ vecs[q]):.2f}")

QUERY: "Game crashes in the first 15 minutes, can't finish making a pawn... even a YEAR after release. This is abysmal. Good luck getting any resolution or bug fixes from Capcom's end, they cheekily have an i"

0.70  "Game crashes a lot in the character creation. Great mechanic, pawns are genius. I really enjoy them, just a bit annoying. Fps in the capital sucks, but in the wilderness it's mostly stable. Adventure "
0.65  'Was having issues with the game crashing during season change, but the release candidate beta patch they have available fixed this issue for me. Having an absolute blast with the game so far.'
0.63  'I bought this game a few days ago and it keeps crashing. Is this a common issue?'
0.63  'After playing this game for 8hrs ATM it is VERY buggy crashes every time you "Run" through different locations and the game is just not very well optimized also the game sometimes just feels extremely'
0.60  'I honestly was excited to play this and it starts off well, my only issue i

8 of the top 10 are about crashes or freezes.